# Problem Set: Formalising a hospital access policy with Claude
### Chapter 2 — First-Order Logic and Reasoning

**Topic coverage:** Keet §2.1 (syntax, semantics, formalising natural language), §2.2 (reasoning, finite models, resolution); course notebooks 01–03 of this chapter. Tools: DSPy, and GEPA through `agentlab.optimize` (new in this chapter); the Chapter 2 model checker.

**Level:** Graduate (practice-first). Combines conceptual analysis with building, measuring and optimising a real Claude-backed agent.

**Total points:** 86 (Part A 20 / Part B 30 / Part C 28 / Part D 8); 47 pts are auto-graded by the check cells, the rest are marked by hand against the rubric. **Estimated effort:** 10–12 hours.

**API budget:** ≈ \$0.65 **measured** for one full run in the reference run (`claude-sonnet-5-5` for the formaliser, `claude-opus-5-5` for GEPA's reflection; ≈ 350 short calls; the GEPA run cost \$0.30). Re-running unchanged cells is free: DSPy caches identical requests. Every model call in this notebook is a billed request to Anthropic; the cost lines the problems ask for are part of the answer.

## The brief

St. Brigid's Hospital is replacing a manual access review with an automated
**compliance checker**. The data-governance office writes its access policy in
English; the checker needs it in first-order logic so it can *decide* whether a
request is permitted. You are the ontology engineer on the project. Your brief:

1. **Build the grader first.** The office will not accept "the model looks right".
   Formalisations are graded by a decision procedure — equivalence over finite
   models — with partial credit and diagnostics that name the mistake.
2. **Build a Claude formaliser** that turns policy sentences into formulas over a
   fixed, reviewed vocabulary, measure it on a development split, and improve it
   with GEPA. Report on a **held-out test split** you never optimised against,
   with the cost of getting there and the run-to-run noise.

The policy corpus (48 sentences, split 16/16/16 by item: 24 written for the course and
24 modelled on HIPAA Privacy Rule provisions on research use, minors and guardians, and
minimum-necessary access) and the predicate vocabulary are provided in `ch02_agentic.py`.
Part D tests the result on a domain it never saw: the requirements of SIP's demo.

> The original brief went on to a **compliance agent** that proves its verdicts and an
> **MDP over proof search**. They return in Chapter 3 (tool-calling agents) and
> Chapter 6 (MDPs), the chapters that introduce those layers.

## Submission requirements

1. **This notebook, executed top to bottom**, with every `TODO` implemented and every *Your answer* cell written.
2. **The artefacts it saves** under `artifacts/` (optimised instructions, run logs) — they are evidence, not by-products.
3. **Honest reporting.** A number without its split, its sample size and its cost earns no credit. Report the runs you did, including the ones that did not improve anything.

The check cells are the marking scheme for the auto-graded points; do not edit them. The final self-test prints your auto-graded subtotal.

> Solutions: `04_solutions.ipynb` — do not open it until you have attempted every problem.

In [ ]:
import sys, json, logging
from pathlib import Path

# Make the course root (the folder holding agentlab/) and this chapter importable.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "agentlab").is_dir():
        sys.path.insert(0, str(candidate))
        break
sys.path.insert(0, str(here))

import agentlab
from agentlab import llm
from agentlab.grading import Grader

print(json.dumps(agentlab.describe_environment(), indent=1))
llm.require_credentials()             # this problem set calls the live Anthropic API
logging.getLogger("dspy").setLevel(logging.WARNING)
GRADER = Grader('Chapter 2 — First-Order Logic and Reasoning — Formalising a hospital access policy with Claude')

import difflib, functools, time
import dspy
import pandas as pd
import ch02_toolkit as fol
import ch02_agentic as A
from agentlab import evaluation as ev, optimize as opt

train, dev, test = A.build_dataset("train"), A.build_dataset("dev"), A.build_dataset("test")
GOLD = {ex.id: fol.parse(ex.gold_formula) for ex in train + dev + test}
print(f"train {len(train)} / dev {len(dev)} / test {len(test)}")
print(A.vocabulary_text())

---

## Part A — Semantic grading and its limits

Before you let a model near the policy, decide how its output will be judged. The
Chapter 2 model checker gives you `A.equivalent(f, g, max_size)`: two closed formulas
agree on **every** interpretation with a domain of at most `max_size` elements.

### A1. Why string similarity cannot grade formalisations (10 pts)

For **three** policy items of your choice, write two alternative formalisations each:

* `"equivalent"` — means the same as the gold formula but is written *differently*
  (not just re-spaced): contraposition, De Morgan, a renamed variable, …;
* `"mistake"` — one of the classic §2.1 errors (conjunction under a universal,
  implication under an existential, negation scope, quantifier order).

Build a DataFrame `a1` with one row per alternative and columns `item`, `kind`,
`formula`, `equivalent` (the checker's verdict against gold) and `string_similarity`
(`difflib.SequenceMatcher` ratio against the gold text). Choose your items so that at
least once the *mistake* is textually closer to gold than the equivalent is.

Then answer in writing: why does this rule out string overlap (BLEU, edit distance)
as the metric for this task, and what does it imply for LLM-as-judge grading here?

In [ ]:
alternatives = {
    # "item-id": {"equivalent": "...", "mistake": "..."},
}
# TODO: fill `alternatives` for three items, then build the DataFrame `a1`.
a1 = None

raise NotImplementedError

In [ ]:
# Marking scheme for A1 -- do not edit.
with GRADER.check('A1', points=6):
    assert isinstance(a1, pd.DataFrame) and len(alternatives) >= 3
    assert {"item", "kind", "formula", "equivalent", "string_similarity"} <= set(a1.columns)
    gold_text = {ex.id: ex.gold_formula for ex in train + dev + test}
    for item, alt in alternatives.items():
        assert A.equivalent(fol.parse(alt["equivalent"]), GOLD[item]), f"{item}: 'equivalent' is not"
        assert not A.equivalent(fol.parse(alt["mistake"]), GOLD[item]), f"{item}: 'mistake' is equivalent"
        assert alt["equivalent"].replace(" ", "") != gold_text[item].replace(" ", ""), \
            f"{item}: rewrite the equivalent, don't copy gold"
    sim = a1.pivot(index="item", columns="kind", values="string_similarity")
    assert (sim["mistake"] > sim["equivalent"]).any(), "find a case where the mistake looks closer"

**Your answer:**

*(Write your answer here.)*

### A2. What the finite-model grader can and cannot see (10 pts)

(a) Using only vocabulary predicates, find a pair of closed formulas `pair` that the
checker calls equivalent at `max_size=1` but not at `max_size=2`. (Hint: think about
the policy items that differ only in quantifier order.)

(b) The checker enumerates all `2**bits` interpretations, where `bits` is
`A.model_space_bits(formulas, size)`. For three gold formulas with different
signatures, build `a2` with columns `item`, `size`, `bits`, `seconds` for sizes 1–4,
timing a fresh self-entailment check (`fol.entails([f], f, size)`) **only when
`bits <= 16`** and recording `None` otherwise.

(c) In writing: which size would you run the production grader at, and why? State
precisely which kind of grading error remains possible at that size, and which cannot
happen.

In [ ]:
pair = ("...", "...")       # TODO (a)

a2 = None                   # TODO (b): one row per (item, size)

raise NotImplementedError

In [ ]:
# Marking scheme for A2 -- do not edit.
with GRADER.check('A2', points=5):
    f, g = (fol.parse(t) for t in pair)
    assert not fol.free_vars(f) and not fol.free_vars(g)
    assert all(p in A.VOCABULARY for p in {**fol.predicates(f), **fol.predicates(g)})
    assert A.equivalent(f, g, max_size=1) and not A.equivalent(f, g, max_size=2)
    assert isinstance(a2, pd.DataFrame) and a2["item"].nunique() >= 3
    assert set(a2["size"]) >= {1, 2, 3, 4}
    for _, grp in a2.sort_values("size").groupby("item"):
        assert list(grp["bits"]) == sorted(grp["bits"]), "bits must grow with the domain size"
        assert all((s is None or s != s) == (b > 16) for s, b in zip(grp["seconds"], grp["bits"]))

**Your answer:**

*(Write your answer here.)*

---

## Part B — Build the grader and the formaliser

The guidelines a scorer can report are in `A.POLICY_RULEBOOK` — ids plus the sentence
GEPA's reflection step will read:

In [ ]:
for rule in A.POLICY_RULEBOOK:
    print(f"{rule.id:30s} {rule.description[:90]}")

### B1. A staged, diagnostic scorer (14 pts)

Implement `policy_scorer(gold, pred, max_size=2, vocabulary=None) -> ev.ScoreReport`
(`vocabulary` defaults to `A.VOCABULARY`; Part D passes another one), where `gold` is a
dataset row (`gold.gold_formula`) and `pred.formula` is the model's answer. Stages, in
this order:

| outcome | score | `violated` |
|---|---|---|
| empty, or `fol.parse` raises | 0.0 | `["emit-parseable-formula"]` |
| free variables | 0.0 | `["close-every-variable"]` |
| a predicate not in the vocabulary, a wrong arity, or any constant | 0.1 | `["use-the-vocabulary"]` |
| well-formed but not equivalent (at `max_size`) | 0.25 | `[A.diagnose(...)]`, or `["match-the-intended-reading"]` when it returns `None` |
| equivalent | 1.0 | `[]` |

Refuse to model-check (score 0.1, `match-the-intended-reading`) when
`A.model_space_bits([pred, gold], max_size) > 16`. For a wrong meaning, put the
produced and intended formulas **and a countermodel** (`fol.find_countermodel(...)
.describe()`, in whichever direction exists) in `notes` — that text is what GEPA reads.

Why the partial credit? A flat zero tells an optimiser only *that* it failed; 0.1 /
0.25 steps give it a gradient: first be parseable, then use the vocabulary, then be
right.

In [ ]:
MAX_BITS = 16

def policy_scorer(gold, pred, max_size: int = 2, vocabulary: dict | None = None) -> ev.ScoreReport:
    text = str(getattr(pred, "formula", "") or "").strip()
    gold_f = fol.parse(gold.gold_formula)
    # TODO: the five stages from the table, in order.
    raise NotImplementedError

In [ ]:
# Marking scheme for B1 -- do not edit.
with GRADER.check('B1', points=12):
    gold = next(ex for ex in dev if ex.id == "nurses-are-clinicians")
    P = lambda text: dspy.Prediction(formula=text)
    cases = [
        ("forall x (Nurse(x) -> Clinician(x))", 1.0, []),
        ("forall y (~Clinician(y) -> ~Nurse(y))", 1.0, []),
        ("", 0.0, ["emit-parseable-formula"]),
        ("The formula is: forall x (Nurse(x) -> Clinician(x)).", 0.0, ["emit-parseable-formula"]),
        ("forall x (Nurse(x) -> Clinician(y))", 0.0, ["close-every-variable"]),
        ("forall x (Nurse(x) -> IsClinician(x))", 0.1, ["use-the-vocabulary"]),
        ("forall x (Nurse(x) -> Treats(x))", 0.1, ["use-the-vocabulary"]),
        ("Nurse(Ana) -> Clinician(Ana)", 0.1, ["use-the-vocabulary"]),
        ("forall x (Nurse(x) & Clinician(x))", 0.25, ["universal-uses-implication"]),
        ("forall x (Clinician(x) -> Nurse(x))", 0.25, ["match-the-intended-reading"]),
    ]
    for text, score, violated in cases:
        r = policy_scorer(gold, P(text))
        assert abs(r.score - score) < 1e-9 and r.violated == violated, (text, r.score, r.violated)
        assert all(v in A.POLICY_RULEBOOK for v in r.violated)
    wrong = policy_scorer(gold, P("forall x (Clinician(x) -> Nurse(x))"))
    assert any("domain" in n for n in wrong.notes), "a wrong reading must come with a countermodel"

### B2. The formaliser as a DSPy program (7 pts)

Write a signature `PolicyFormalisation` with inputs `statement` and `vocabulary` and
output `formula`, and a factory `PolicyFormaliser(instruction)` returning a
`dspy.Module` with a single `dspy.Predict` whose instruction is `instruction`. The
field descriptions are part of the prompt — write them as you would brief a colleague
(manual marks). Then configure Claude and run the program once on `train[0]`.

In [ ]:
BASELINE_INSTRUCTION = "Formalise the policy sentence in first-order logic."

# TODO: class PolicyFormalisation(dspy.Signature): ...
# TODO: def PolicyFormaliser(instruction=BASELINE_INSTRUCTION): ...

lm = llm.configure_dspy()
smoke = None     # TODO: PolicyFormaliser()(**train[0].inputs())

raise NotImplementedError

In [ ]:
# Marking scheme for B2 -- do not edit.
with GRADER.check('B2', points=4):
    assert set(PolicyFormalisation.input_fields) == {"statement", "vocabulary"}
    assert "formula" in PolicyFormalisation.output_fields
    program = PolicyFormaliser("custom instruction")
    assert len(list(program.named_predictors())) == 1
    assert opt.instruction_of(program) == "custom instruction"
    assert isinstance(smoke.formula, str) and smoke.formula.strip()

### B3. Baseline on the development split, with its cost (9 pts)

Evaluate the baseline program on `dev` with `ev.evaluate_dataset`, inside
`llm.meter(lm)` so you know what it cost. Store the result in `baseline_dev` and the
cost in `baseline_dev_cost`. Print the per-item rows.

Then write a short **error analysis**: for each non-perfect item, which guideline was
violated and *why* you think Claude made that choice. Separate genuine logic errors from
vocabulary/format problems — they call for different fixes.

In [ ]:
# TODO: baseline_dev = ..., baseline_dev_cost = ...

raise NotImplementedError

In [ ]:
# Marking scheme for B3 -- do not edit.
with GRADER.check('B3', points=3):
    assert baseline_dev["n"] == len(dev) == 16
    assert {r["item"] for r in baseline_dev["rows"]} == {ex.id for ex in dev}
    assert 0.0 <= baseline_dev["mean_score"] <= 1.0
    assert {"calls", "usd"} <= set(baseline_dev_cost)

**Your answer:**

*(Write your answer here.)*

---

## Part C — Optimise with GEPA — and report it honestly

GEPA rewrites the instruction by reflecting on the scorer's feedback. It is only as
good as that feedback, it overfits small training sets, and every metric call is a
billed request. The rules for this part: **optimise on `train`, select on `dev`,
report on `test`** — and put a cost and a noise estimate next to every number.

**First, check that there is something to optimise.** The B2 formaliser, with its
careful field descriptions, is already at or near the ceiling on a current model (the
live run scored 1.0 on `dev`); GEPA then has no failures to reflect on and returns the
instruction unchanged. Part C therefore optimises the formaliser a team typically ships
*first*: `BareFormaliser`, the same task with a bare signature and no syntax hints. On
`claude-sonnet-5-5` it writes correct logic in Unicode notation (`∀x (Clinician(x) →
Employee(x))`), and none of it parses. Whether GEPA can recover the output contract from
the scorer's feedback alone is the question.

In [ ]:
class BareFormalisation(dspy.Signature):
    """Formalise the policy sentence in first-order logic."""

    statement: str = dspy.InputField()
    vocabulary: str = dspy.InputField()
    formula: str = dspy.OutputField()


def BareFormaliser(instruction: str = BASELINE_INSTRUCTION):
    """The first version a team ships: the task, and nothing about the output syntax."""
    class _Bare(dspy.Module):
        def __init__(self):
            super().__init__()
            self.formalise = dspy.Predict(BareFormalisation.with_instructions(instruction))

        def forward(self, statement: str, vocabulary: str):
            return self.formalise(statement=statement, vocabulary=vocabulary)

    return _Bare()


with llm.meter(lm) as headroom_cost:
    headroom = ev.evaluate_dataset(BareFormaliser(), dev, policy_scorer)
print("bare formaliser on dev:", headroom["mean_score"], headroom["violations"], headroom_cost)

### C1. A budgeted GEPA run with a held-out report (12 pts)

1. Build the GEPA feedback metric from your scorer and `A.POLICY_RULEBOOK`, and a
   separate reflection LM (`llm.reflection_lm()`).
2. Run `opt.run_gepa` on `train` with `valset=dev` and `max_metric_calls=GEPA_BUDGET`
   inside `llm.meter(lm, reflect)`; store the cost in `gepa_cost`.
3. Compare the bare baseline (`BareFormaliser()`) and the tuned program on **`test`**
   with `opt.compare`
   (store as `c1`) and print `c1.report()`.
4. Save the tuned instruction to `agentlab.artifacts_dir() /
   "ch02_policy_formaliser_instruction.txt"` — it is a learned artefact and belongs
   under review like code.

In writing: read the instruction diff. Which guidelines did GEPA put into words, which
did it miss, and did it add anything that is *not* a guideline (over-fitting to the
sixteen training sentences)?

In [ ]:
GEPA_BUDGET = 60
# TODO: gepa_metric, reflect, tuned (inside llm.meter -> gepa_cost), c1, save the instruction
instruction_path = agentlab.artifacts_dir() / "ch02_policy_formaliser_instruction.txt"

raise NotImplementedError

In [ ]:
# Marking scheme for C1 -- do not edit.
with GRADER.check('C1', points=6):
    ids = lambda rows: {ex.id for ex in rows}
    assert not (ids(test) & (ids(train) | ids(dev))), "the test split must be untouched"
    assert {r["item"] for r in c1.after["rows"]} == ids(test), "report on the test split"
    assert c1.before["n"] == c1.after["n"] == len(test)
    assert instruction_path.is_file()
    assert instruction_path.read_text(encoding="utf-8") == opt.instruction_of(tuned)
    assert {"calls", "usd"} <= set(gepa_cost) and GEPA_BUDGET <= 200

**Your answer:**

*(Write your answer here.)*

### C2. Was the optimiser worth it? Three programs, one table (8 pts)

Add a third contender: the **hand-written guidelines** —
`BareFormaliser(A.POLICY_RULEBOOK.as_guidelines(BASELINE_INSTRUCTION))`. Evaluate baseline, guidelines
and GEPA-tuned on `test`, each inside its own `llm.meter`, and build `c2` with columns
`program` (`"baseline"`, `"guidelines"`, `"gepa"`), `test_mean`, `violations`,
`eval_usd`, and `optimisation_usd` (the GEPA cost for `"gepa"`, 0 otherwise).

In writing: which would you deploy, and what would change your mind? Consider score,
per-call token cost (instruction length), optimisation cost, and maintenance.

In [ ]:
guidelines_instruction = A.POLICY_RULEBOOK.as_guidelines(BASELINE_INSTRUCTION)
c2 = None    # TODO

raise NotImplementedError

In [ ]:
# Marking scheme for C2 -- do not edit.
with GRADER.check('C2', points=3):
    assert isinstance(c2, pd.DataFrame)
    assert set(c2["program"]) == {"baseline", "guidelines", "gepa"}
    assert {"test_mean", "violations", "eval_usd", "optimisation_usd"} <= set(c2.columns)
    assert c2.set_index("program").loc[["baseline", "guidelines"], "optimisation_usd"].eq(0).all()

**Your answer:**

*(Write your answer here.)*

### C3. Is the difference bigger than the noise? (8 pts)

Sixteen test items and a sampling temperature of 1.0: one item moves the mean by up to
0.06. Estimate the noise directly. With caching **disabled**
(`fresh = llm.dspy_lm(cache=False)` and `with dspy.context(lm=fresh): ...`), run the
bare baseline (`BareFormaliser()`) and the GEPA-tuned programs on `test` three times each; store the mean scores
in `runs = {"baseline": [..3..], "gepa": [..3..]}`.

Compute each program's mean and sample standard deviation and set `verdict_c3` to
`"significant"` if `|mean_gepa - mean_baseline| > 2 * max(sd_gepa, sd_baseline)`,
else `"not significant"`. In writing: what does this say about the C1 result, and what
would a credible evaluation of this system need?

In [ ]:
fresh = llm.dspy_lm(cache=False)
runs = {"baseline": [], "gepa": []}
# TODO: fill runs, then compute verdict_c3
verdict_c3 = None

raise NotImplementedError

In [ ]:
# Marking scheme for C3 -- do not edit.
with GRADER.check('C3', points=4):
    import statistics as _st
    assert set(runs) == {"baseline", "gepa"} and all(len(v) == 3 for v in runs.values())
    assert all(0.0 <= x <= 1.0 for v in runs.values() for x in v)
    _m = {k: _st.mean(v) for k, v in runs.items()}
    _s = {k: _st.stdev(v) for k, v in runs.items()}
    _expected = ("significant" if abs(_m["gepa"] - _m["baseline"]) > 2 * max(_s.values())
                 else "not significant")
    assert verdict_c3 == _expected

**Your answer:**

*(Write your answer here.)*

---

## Part D — Does it transfer? The Semantic Intelligence Platform's requirements

A formaliser is worth tuning only if what it learned is *logic*, not *hospital*. The
Semantic Intelligence Platform (SIP, `sip/` in this repository) turns a project brief
into competency questions; its demo brief and source document contain exactly the kind
of sentences you have been formalising, in a different domain. `A.build_sip_dataset()`
holds seven of them, verbatim, with gold formulas over a fleet vocabulary
(`A.SIP_VOCABULARY`).

In [ ]:
sip = A.build_sip_dataset()
for ex in sip:
    print(f"{ex.statement:70s} {ex.gold_formula}")
print()
print(A.sip_vocabulary_text())

### D1. Does the tuned formaliser transfer? SIP's requirements (8 pts)

Grade with your B1 scorer, pointed at the fleet vocabulary:
`sip_scorer = functools.partial(policy_scorer, vocabulary=A.SIP_VOCABULARY)`. Evaluate
the three C2 contenders (`"baseline"`, `"guidelines"`, `"gepa"`) on `sip`, each inside
its own `llm.meter(lm)`, and build `d1` with columns `program`, `sip_mean`,
`violations`, `eval_usd`.

In writing: (a) which lines of the GEPA instruction are about *logic* (they should
transfer) and which are about the *hospital* (they cannot)? (b) Does the ranking of the
three programs on `sip` match their ranking on `test`? (c) SIP's RequirementsAgent writes
competency questions as SPARQL tests; what would you change before using this
formaliser to check that a CQ's SPARQL says what its English sentence says?

#### Background and design decisions — D1

A held-out *test split* measures generalisation to new sentences from the same policy.
A *transfer set* measures generalisation to a new domain — the question that decides
whether an optimised instruction is a reusable asset or a fitted one. Seven items is
small; read the per-item rows, not only the mean.

Decisions you make:

* **Same grader, new vocabulary.** The scorer's logic does not change; only the
  vocabulary does. That is why B1 takes `vocabulary` as a parameter.
* **No re-tuning.** Run the programs exactly as C1 and C2 left them: re-tuning on the
  transfer set would make it a training set.

#### Contract — D1

| name | type | content |
|---|---|---|
| `sip_scorer` | callable | `functools.partial(policy_scorer, vocabulary=A.SIP_VOCABULARY)` |
| `results_d1` | `dict[str, dict]` | the three `ev.evaluate_dataset(program, sip, sip_scorer)` results, keyed by program name |
| `d1` | `pd.DataFrame` | 3 rows; columns `program`, `sip_mean`, `violations`, `eval_usd` |

In [ ]:
sip_scorer = None      # TODO: functools.partial(policy_scorer, vocabulary=A.SIP_VOCABULARY)
results_d1 = {}
d1 = None
raise NotImplementedError

In [ ]:
# Marking scheme for D1 -- do not edit.
with GRADER.check('D1', points=4):
    assert isinstance(d1, pd.DataFrame) and set(d1["program"]) == {"baseline", "guidelines", "gepa"}
    assert {"sip_mean", "violations", "eval_usd"} <= set(d1.columns)
    for name in ("baseline", "guidelines", "gepa"):
        assert {r["item"] for r in results_d1[name]["rows"]} == {ex.id for ex in sip}
    gold_pred = dspy.Prediction(formula=sip[0].gold_formula)
    assert sip_scorer(sip[0], gold_pred).score == 1.0, "a SIP gold formula must score 1.0"
    assert policy_scorer(sip[0], gold_pred).violated == ["use-the-vocabulary"], \
        "the default vocabulary is still the hospital's"

**Your answer:**

*(Write your answer here.)*

---

## Grading rubric

| Part | Problem | Auto | Manual | Pts |
|---|---|---|---|---|
| A | A1 Why string similarity cannot grade formalisations | 6 | 4 | 10 |
| A | A2 What the finite-model grader can and cannot see | 5 | 5 | 10 |
| B | B1 A staged, diagnostic scorer | 12 | 2 | 14 |
| B | B2 The formaliser as a DSPy program | 4 | 3 | 7 |
| B | B3 Baseline on the development split, with its cost | 3 | 6 | 9 |
| C | C1 A budgeted GEPA run with a held-out report | 6 | 6 | 12 |
| C | C2 Was the optimiser worth it? Three programs, one table | 3 | 5 | 8 |
| C | C3 Is the difference bigger than the noise? | 4 | 4 | 8 |
| D | D1 Does the tuned formaliser transfer? SIP's requirements | 4 | 4 | 8 |
| **Total** | | | | **86** |

*Manual* points are for the written answers and for the quality of the engineering evidence (splits, sample sizes, cost, honest negative results).

---

## Self-test

Re-run this cell after finishing. It lists every auto-graded problem and whether its check passed.

In [ ]:
AUTO_GRADED = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3', 'D1']
_ = GRADER.summary(AUTO_GRADED)